# Phase 1: YOLOX ROI extraction on the complete RSNA dataset (dual T4, 15 batches)
Kaggle, Internet ON, GPU ON. Inputs to attach: the RSNA competition + the dataset `dangnh0611/rsna-breast-cancer-detection-best-ckpts`.

All rows in `train.csv` are processed. The 15 batches are execution/checkpointing units only — **not** train/validation/test splits and not sampling. Both T4 GPUs work concurrently. Each GPU loads YOLOX once and keeps it resident while processing its 15 shards. DICOM prefetch is increased to 8 reader threads per GPU; inference behavior remains unchanged (one image per YOLOX forward) to preserve the existing ROI logic.


In [ ]:
import os
if not os.path.isdir("/kaggle/working/mammogram-cancer-classifier"):
    !git clone https://github.com/hassenmohamed-mezr/mammogram-cancer-classifier.git /kaggle/working/mammogram-cancer-classifier
%cd /kaggle/working/mammogram-cancer-classifier
!git pull
!pip -q install pydicom pylibjpeg pylibjpeg-openjpeg pylibjpeg-libjpeg
# if DICOM decoding fails later:  !pip -q install python-gdcm

### Add YOLOX (winner's breast detector)
1. Right sidebar -> **Add Input** -> search `rsna-breast-cancer-detection-best-ckpts` (by dangnh0611) -> add.
2. Run the next cell: installs YOLOX's small dependencies, clones the winner's repo (code only, ~80 MB, MIT licence) and finds the checkpoint.

In [ ]:
!pip -q install loguru thop pycocotools
WINNER = "/kaggle/working/rsna_winner"
if not os.path.isdir(WINNER):
    !git clone --depth 1 -b reproduce https://github.com/dangnh0611/kaggle_rsna_breast_cancer.git {WINNER}

import glob
hits = glob.glob("/kaggle/input/**/yolox_nano_416_roi_torch.pth", recursive=True)
print("checkpoint:", hits)
assert hits, "Add the best-ckpts dataset as an input (see step 1)"
CKPT = hits[0]

In [ ]:
!ls /kaggle/input
!ls /kaggle/input/competitions/rsna-breast-cancer-detection

In [ ]:
# ---- EDIT only if the ls above shows different names ----
COMP     = "/kaggle/input/competitions/rsna-breast-cancer-detection"
CSV      = f"{COMP}/train.csv"
IMG_ROOT = f"{COMP}/train_images"
IMG_TMPL = "{root}/{patient_id}/{image_id}.dcm"
OUT      = "/kaggle/working/all_images_manifest"


In [ ]:
# Build one manifest from every row in train.csv. No split and no sampling.
import pandas as pd
from pathlib import Path

Path(OUT).mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(CSV)
manifest["path"] = [
    IMG_TMPL.format(root=IMG_ROOT, patient_id=p, image_id=i)
    for p, i in zip(manifest["patient_id"], manifest["image_id"])
]
manifest.to_csv(f"{OUT}/roi_manifest.csv", index=False)


In [ ]:
print("Images in complete manifest:", len(manifest))
print("Class counts:")
print(manifest["cancer"].value_counts().sort_index().rename(index={0: "class_0", 1: "class_1"}))


In [ ]:
# YOLOX instance for the existing visual preview cell (GPU 0).
# The full extraction below launches one independent YOLOX process per GPU.
from src.dicom_roi import WinnerRoi, load_dicom, process
roi = WinnerRoi(WINNER, CKPT, device="cuda:0")
print("preview device:", roi.device)


In [ ]:
import numpy as np, matplotlib.pyplot as plt

rows = pd.concat([
    manifest[manifest.cancer == 1].sample(10, random_state=1),
    manifest[manifest.cancer == 0].sample(10, random_state=1)
])
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
    crop, info = process(load_dicom(r.path), roi)
    conf = "" if info["conf"] is None else f" {info['conf']:.2f}"
    ax.imshow(crop, cmap="gray"); ax.axis("off")
    ax.set_title(f"cancer={r.cancer} | {info['source']}{conf} | {crop.shape[1]}x{crop.shape[0]}", fontsize=9)
plt.tight_layout(); plt.show()


**Check the grid:** most titles should say `yolo`, the whole breast must be inside each crop, no inverted images.
If many say `otsu`/`full`, or boxes are wrong, stop and send me the grid.


## Full-dataset ROI extraction: 15 resumable batches across both T4 GPUs
The 54,706 manifest rows are divided into 15 sequential batches of approximately 3,647 images. Within each batch, rows are split across GPU 0 and GPU 1. The two GPU worker processes run concurrently and each loads its model only once.

Each GPU uses `prefetch=8` DICOM reader threads to overlap decoding with inference. The existing YOLOX `detect()` still performs one-image inference; true batched model inference requires a separate, parity-tested change to `WinnerRoi.detect()` and is intentionally not enabled here.

Every batch writes its own `roi_results.csv` and crops. The existing `run()` function skips image IDs already recorded in that batch output, so rerunning this cell resumes completed work instead of processing it again. Stop any currently running old extraction before starting this revised cell.


In [ ]:
# One persistent process per GPU. Each process loads YOLOX once, then runs all 15 batch shards.
from pathlib import Path

WORKER_SCRIPT = "/kaggle/working/run_roi_gpu_worker_15batches.py"
Path(WORKER_SCRIPT).write_text(r'''import argparse
import sys
import torch
from pathlib import Path

sys.path.insert(0, "/kaggle/working/mammogram-cancer-classifier")
from src.dicom_roi import WinnerRoi
from src.run_roi import run

parser = argparse.ArgumentParser()
parser.add_argument("--gpu", type=int, required=True)
parser.add_argument("--manifest-root", required=True)
parser.add_argument("--output-root", required=True)
parser.add_argument("--winner", required=True)
parser.add_argument("--checkpoint", required=True)
parser.add_argument("--num-batches", type=int, default=15)
parser.add_argument("--prefetch", type=int, default=8)
args = parser.parse_args()

torch.cuda.set_device(args.gpu)
device = f"cuda:{args.gpu}"
roi = WinnerRoi(args.winner, args.checkpoint, device=device)
print(f"[GPU {args.gpu}] model ready on {roi.device}", flush=True)

manifest_root = Path(args.manifest_root)
output_root = Path(args.output_root)
for batch_id in range(args.num_batches):
    batch_name = f"batch_{batch_id + 1:02d}"
    manifest_path = manifest_root / batch_name / f"gpu{args.gpu}_manifest.csv"
    batch_out = output_root / batch_name / f"gpu{args.gpu}_output"
    print(f"[GPU {args.gpu}] starting {batch_name}: {manifest_path}", flush=True)
    run(
        str(manifest_path),
        str(batch_out),
        roi=roi,
        prefetch=args.prefetch,
        flush_every=200,
    )
    print(f"[GPU {args.gpu}] completed {batch_name}", flush=True)
''' , encoding="utf-8")


In [ ]:
# Create 15 execution batches, split each batch across both GPUs, then launch two persistent workers.
# This is NOT a train/validation/test split: every manifest row is retained exactly once.
import os, sys, subprocess
from pathlib import Path
import numpy as np

NUM_BATCHES = 15
PREFETCH = 8  # DICOM read/decode workers per GPU; reduce to 4 if host RAM becomes constrained.
ROI_OUT = "/kaggle/working/roi_out"
WORKER_ROOT = Path("/kaggle/working/roi_workers_15batches")
WORKER_ROOT.mkdir(parents=True, exist_ok=True)

full_manifest = pd.read_csv(f"{OUT}/roi_manifest.csv")
index_batches = np.array_split(np.arange(len(full_manifest)), NUM_BATCHES)

for batch_idx, indices in enumerate(index_batches, start=1):
    batch_dir = WORKER_ROOT / f"batch_{batch_idx:02d}"
    batch_dir.mkdir(parents=True, exist_ok=True)
    batch_df = full_manifest.iloc[indices].reset_index(drop=True)
    # Interleaved split balances rows and keeps every row exactly once.
    batch_df.iloc[0::2].to_csv(batch_dir / "gpu0_manifest.csv", index=False)
    batch_df.iloc[1::2].to_csv(batch_dir / "gpu1_manifest.csv", index=False)
    print(f"Batch {batch_idx:02d}: {len(batch_df):,} total | "
          f"GPU0={len(batch_df.iloc[0::2]):,} | GPU1={len(batch_df.iloc[1::2]):,}")

processes = []
for gpu in (0, 1):
    command = [
        sys.executable, WORKER_SCRIPT,
        "--gpu", str(gpu),
        "--manifest-root", str(WORKER_ROOT),
        "--output-root", str(WORKER_ROOT),
        "--winner", WINNER,
        "--checkpoint", CKPT,
        "--num-batches", str(NUM_BATCHES),
        "--prefetch", str(PREFETCH),
    ]
    log_path = WORKER_ROOT / f"gpu{gpu}.log"
    log_file = open(log_path, "a", buffering=1)
    proc = subprocess.Popen(
        command,
        cwd="/kaggle/working/mammogram-cancer-classifier",
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    processes.append((gpu, proc, log_file, log_path))
    print(f"Started GPU {gpu}: PID={proc.pid} | log={log_path}")

failed = []
for gpu, proc, log_file, log_path in processes:
    return_code = proc.wait()
    log_file.close()
    print(f"GPU {gpu} worker exit code: {return_code} | log: {log_path}")
    if return_code != 0:
        failed.append(gpu)

if failed:
    raise RuntimeError(f"GPU worker(s) failed: {failed}. Inspect their log files before rerunning; completed images are resumable.")


In [ ]:
# Merge all 30 worker-batch outputs into class_0/class_1 and a unified results table.
# Re-running this merge is safe: it copies/moves only files still present in batch crop folders.
import shutil
from pathlib import Path

roi_root = Path(ROI_OUT)
class_0_dir = roi_root / "class_0"
class_1_dir = roi_root / "class_1"
class_0_dir.mkdir(parents=True, exist_ok=True)
class_1_dir.mkdir(parents=True, exist_ok=True)

result_parts = []
error_parts = []
missing_outputs = []
for batch_idx in range(1, NUM_BATCHES + 1):
    for gpu in (0, 1):
        worker_out = WORKER_ROOT / f"batch_{batch_idx:02d}" / f"gpu{gpu}_output"
        results_path = worker_out / "roi_results.csv"
        if not results_path.exists():
            missing_outputs.append(str(results_path))
            continue
        part = pd.read_csv(results_path)
        for idx, row in part.iterrows():
            destination_dir = class_1_dir if int(row.cancer) == 1 else class_0_dir
            destination = destination_dir / f"{row.image_id}.png"
            source = Path(row.roi_path)
            if source.exists():
                if destination.exists():
                    source.unlink()
                else:
                    shutil.move(str(source), str(destination))
            elif not destination.exists():
                raise FileNotFoundError(f"ROI file missing in both source and destination: {source}")
            part.at[idx, "roi_path"] = str(destination)
        result_parts.append(part)

        errors_path = worker_out / "roi_errors.csv"
        if errors_path.exists():
            error_parts.append(pd.read_csv(errors_path))

if missing_outputs:
    raise RuntimeError("Some batch outputs are missing; do not treat extraction as complete:\n" + "\n".join(missing_outputs[:10]))

results = pd.concat(result_parts, ignore_index=True) if result_parts else pd.DataFrame()
results = results.drop_duplicates(subset=["image_id"], keep="last")
results.to_csv(roi_root / "roi_results.csv", index=False)
if error_parts:
    errors = pd.concat(error_parts, ignore_index=True).drop_duplicates(subset=["image_id"], keep="last")
    errors.to_csv(roi_root / "roi_errors.csv", index=False)
else:
    errors = pd.DataFrame()

print("Manifest images:", len(full_manifest))
print("Extracted result rows:", len(results))
print("Class 0:", int((results.cancer == 0).sum()) if len(results) else 0)
print("Class 1:", int((results.cancer == 1).sum()) if len(results) else 0)
print("Errors recorded:", len(errors))
print("Unique image IDs:", results.image_id.nunique() if len(results) else 0)
print("Output:", ROI_OUT)


**After the full run:** save `/kaggle/working/roi_out` as a Kaggle Dataset. It contains `class_0/`, `class_1/`, `roi_results.csv`, and (if any images failed) `roi_errors.csv`.

The 15 batch manifests, resumable batch outputs, and per-GPU logs are under `/kaggle/working/roi_workers_15batches/`. If a session stops, rerun cells 12–14 after the environment is restored; the `run()` function skips image IDs already recorded in each batch output.
